In [1]:
import pandas as pd
import re
from IPython.display import display, HTML


In [55]:
df = pd.read_csv(r'C:\Users\joly-\Github\HUMAN\news\csv\AI_bert_2025_seed_8702_strict.csv')
df.head()

,Unnamed: 0,title,outlet,date,authors,body,word_count,ai_related,matched_keywords_title,matched_keywords_body,...,n_hits_body_total,company_hits,processed,nouns,adjectives,verbs,date_parsed,year,topic,probability
0,2746,kruiswoordtest 6821 kruiswoordtest 6821,TR,2025-02-21,JAAP DE BERG,"Horizontaal1mediabedrijf met Bild en Die Welt,...",146,yes,[],"['ai', 'openai']",...,2,[],horizontaal1mediabedrijf atleet sprong spreekt...,horizontaal1mediabedrijf atleet sprong spreekt...,figuurlijk aanzienlijk ander 18verhaal Russisc...,leveren maken korten 10tevred 14buren kennen l...,2025-02-21,2025,22,0.019634
1,2750,'Ik ben toch niet de enige die op Netflix denk...,TR,2025-05-24,ELLEKE BAL,'Houd je stem laag'. Zo vertaalt Google Transl...,270,yes,[],['ai'],...,1,['google'],stem zin machinevertaler metaforen mist lettie...,stem zin machinevertaler metaforen mist lettie...,laag simpel engels letterlijk deepl goed hard ...,houden vertalen kunnen doen komen praten gaan ...,2025-05-24,2025,36,0.032375
2,2751,Een beter leven zorgt voor meer vertrouwen Een...,TR,2025-01-15,KIM PUTTERS,Er is veel grilligheid in het vertrouwen richt...,281,yes,[],"['ai', 'artificiële intelligentie']",...,2,[],grilligheid vertrouwen richting politiek kabin...,grilligheid vertrouwen richting politiek kabin...,praktisch tijdelijk oud laag continu langlopen...,zijn aantreden schuiven stijgen opvallen zien ...,2025-01-15,2025,83,1.000000
3,2755,iPhone-maker doet goede zaken met AI-servers i...,TR,2025-01-06,Unknown Authors,De Taiwanese elektronicafabrikant Foxconn blij...,116,yes,['ai'],"['ai', 'kunstmatige intelligentie']",...,2,['apple'],elektronicafabrikant foxconn opkomst intellige...,elektronicafabrikant foxconn opkomst intellige...,taiwanees kunstmatig ander voorbij aanzienlijk...,blijven profiteren zetten gaan omgerekend verw...,2025-01-06,2025,14,0.011361
4,2763,En daar gaan de Britse kroonjuwelen En daar ga...,TR,2025-02-27,ILYAZ NASRULLAH,Op Spotify is sinds dinsdag het protestalbum I...,315,yes,[],"['ai', 'chatgpt', 'midjourney']",...,7,[],spotify protestalbum album samenwerking arties...,spotify protestalbum album samenwerking arties...,this Brits this creatief Brits creatief gratis...,beluisteren bestaan horen durend verlaten zet ...,2025-02-27,2025,6,0.094306


In [4]:
df.shape

(298, 22)

In [71]:
# -----------------------------
# SETTINGS
# -----------------------------
topic_id = 78 # choose topic
n_examples = 31  # number of rows to display
text_col = "body"  # or "text_full", etc.

# -----------------------------
# YOUR KEYWORDS
# -----------------------------
keywords = (
    "kunstmatige intelligentie|artificial intelligence|artificiële intelligentie|AI|generatieve AI|"
    "generatieve kunstmatige intelligentie|generatieve artificiële intelligentie|"
    "machine learning|machinaal leren|diep leren|deep learning|neurale netwerken|"
    "large language model|grote taalmodel*|LLM|chatbot*|GPT|ChatGPT|Bard|Claude|"
    "Gemini|mistral|perplexity|ollama|LLaMA|openai|anthropic|midjourney|hugging face|"
    "slimme algoritme*|automatische besluitvorming|automatisch beslissysteem|"
    "algoritmische besluitvorming|algoritme*|cognitieve technologie*|AI-technologie*|"
    "AI-systeem*|AI-toepassing*|AI-model*|spraakherkenning|beeldherkenning|"
    "computer vision|natuurlijke taalverwerking|natural language processing|NLP|robot|drones|drone|grok|xai|deepmind|azure"
)

_COMPANY_NAMES = [
    "NVIDIA", "Apple", "Microsoft", "Google", "Alphabet",
    "Meta Platforms", "Facebook", "Tesla", "Oracle",
    "Palantir", "IBM", "Adobe", "Cambricon Technologies",
    "CoreWeave", "Fermi Inc", "Dynatrace", "Tempus AI",
    "SenseTime", "Mobileye", "Aurora Innovation", "UiPath",
    "SoundHound AI", "ASML", "NXP Semiconductors",
    "BE Semiconductor Industries", "ASM International",
    "Adyen", "Just Eat Takeaway", "Booking.com", "Mollie",
    "Picnic", "TomTom", "Swapfiets", "TKH Group",
    "Ordina", "Nedap", "CM.com", "ICT Group",
    "Neways Electronics", "Ctac", "Photon Energy",
    "Almunda Professionals", "Samsung", "Huawei",
    "Sony", "LG", "Baidu", "Tencent",
    "Alibaba", "Douyin", "Cloudflare",
    "Snowflake", "Docker", "Red Hat",
    "Uber", "Bolt", "Grab", "Epic Games",
    "Unity", "Discord", "Twitter", "X"
]

# -----------------------------
# BUILD REGEX
# -----------------------------

def wildcard_to_regex(pattern):
    return pattern.replace("*", r"\w*")

keyword_patterns = [wildcard_to_regex(k) for k in keywords.split("|")]
company_patterns = [re.escape(c) for c in _COMPANY_NAMES]

all_patterns = keyword_patterns + company_patterns

regex = re.compile(r"\b(" + "|".join(all_patterns) + r")\b", re.IGNORECASE)

# -----------------------------
# HIGHLIGHT FUNCTION
# -----------------------------

def highlight_text(text):
    if pd.isna(text):
        return ""
    
    def repl(match):
        return f"<span style='color:red; font-weight:bold; font-size:120%;'>{match.group(0)}</span>"
    
    return regex.sub(repl, text)

# -----------------------------
# FILTER + SAMPLE
# -----------------------------

df_topic = df[df["topic"] == topic_id].copy()
df_sample = df_topic.sample(n=min(n_examples, len(df_topic)), random_state=42)

# -----------------------------
# DISPLAY
# -----------------------------

for i, row in df_sample.iterrows():
    title = row.get("title", "")
    text = row.get(text_col, "")

    # also show filename, program and year  
    filename = row.get("filename", "")
    program = row.get("program", "")
    year = row.get("year", "")
    meta_info = f"<small><em>{filename} | {program} | {year}</em></small>"
    
    
    html = f"""
    <div style="margin-bottom:30px;">
        {meta_info}
        <h4>{highlight_text(title)}</h4>
        <p>{highlight_text(text)}</p>
    </div>
    """
    
    display(HTML(html))

In [ ]:

# optional: save to Excel
# df_topic.to_excel(f"topic_{topic_id}_subset.xlsx", index=False)